In [1]:
import sys
sys.path.insert(0,'..')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import pandas as pd
import numpy as np
import os
import tensorflow.keras as keras
import tensorflow.keras.backend as K
from scipy.stats import spearmanr
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from source.version2.data import dataLoader
from source.version2.model import BERTModel

In [3]:
def evaluateModel(labels, scores):
    correls = []
    for idx in range(30):
        label = labels.iloc[:,idx]
        score = scores.iloc[:,idx]
        correl = spearmanr(label, score).correlation
        correls.append(correl)
    metric = round(np.mean(correls), 4)
    print('Metric:', metric)
    return None

In [4]:
def score(fold):
    train, valid = dataLoader(fold)
    model = BERTModel()
    path = '../../model/version-2/fold-{}/model.pt'.format(fold)
    model = load_model(path)
    scores = model.predict(valid, verbose = 0)
    labels = []
    for _, label in valid:
        labels.append(label)
    labels = np.vstack(labels)
    scores = pd.DataFrame(scores)
    labels = pd.DataFrame(labels)
    scores.to_csv('../../model/version-2/fold-{}/scores.csv'.format(fold), index=False)
    labels.to_csv('../../model/version-2/fold-{}/labels.csv'.format(fold), index=False)
    evaluateModel(labels, scores)
    del model
    K.clear_session()
    return None

In [5]:
score(1)

Metric: 0.384


In [6]:
score(2)

Metric: 0.3746


In [7]:
score(3)

Metric: 0.3978


In [8]:
score(4)

Metric: 0.3882


In [9]:
score(5)

Metric: 0.3738


In [10]:
score_1 = pd.read_csv('../../model/version-2/fold-1/scores.csv', header=None)
score_2 = pd.read_csv('../../model/version-2/fold-2/scores.csv', header=None)
score_3 = pd.read_csv('../../model/version-2/fold-3/scores.csv', header=None)
score_4 = pd.read_csv('../../model/version-2/fold-4/scores.csv', header=None)
score_5 = pd.read_csv('../../model/version-2/fold-5/scores.csv', header=None)
scores = score_1.append(score_2).append(score_3).append(score_4).append(score_5)

In [11]:
label_1 = pd.read_csv('../../model/version-2/fold-1/labels.csv', header=None)
label_2 = pd.read_csv('../../model/version-2/fold-2/labels.csv', header=None)
label_3 = pd.read_csv('../../model/version-2/fold-3/labels.csv', header=None)
label_4 = pd.read_csv('../../model/version-2/fold-4/labels.csv', header=None)
label_5 = pd.read_csv('../../model/version-2/fold-5/labels.csv', header=None)
labels = label_1.append(label_2).append(label_3).append(label_4).append(label_5)

In [12]:
evaluateModel(labels, scores)

Metric: 0.3778


In [13]:
scores.to_csv('../../model/version-2/scores.csv', index=False)
labels.to_csv('../../model/version-2/labels.csv', index=False)